# Bulk viscosity — Sanchit (TensorFlow DNN from MLQCD)

Reproduces `CODE OF THE AUTHOR/.../Emulator/Bulk_New.ipynb` using **your** plain DNN from `MLQCD.ipynb` (TensorFlow, **not** PyTorch ResNet).

**Prerequisite:** Train in `MLQCD.ipynb`, then save weights:
```python
model.save_weights("mlqcd_model.weights.h5")
```

In [ ]:
# Block 1 — imports (TensorFlow, not PyTorch)
import os
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from scipy.integrate import quad
from sklearn.model_selection import train_test_split

os.makedirs("plots", exist_ok=True)
GEV_TO_INV_FM = 5.07
Tc = 0.155
Nc = 3
Nf = 3

# #region agent log
_LOG_PATH = os.path.join(os.getcwd(), "debug-5a2bf7.log")
def _dblog(msg, data, hyp, loc="Block1"):
    entry = {"sessionId":"5a2bf7","timestamp":int(time.time()*1000),
             "location":f"Bulk_Sanchit.ipynb:{loc}","message":msg,"data":data,"hypothesisId":hyp}
    with open(_LOG_PATH,"a") as _f: _f.write(json.dumps(entry)+"\n")
_dblog("imports ok", {"cwd": os.getcwd()}, "init")
# #endregion

In [ ]:
# Block 2 — YOUR DNN architecture (same as MLQCD.ipynb)
def build_mass_model():
    """Simple non-ResNet DNN: (T, MuB) -> (mg, mu, ms)."""
    inputs = tf.keras.layers.Input(shape=(2,), name="thermo_inputs")
    x = tf.keras.layers.Dense(64, activation="swish", kernel_initializer="he_uniform", name="dense_64_1")(inputs)
    x = tf.keras.layers.Dense(64, activation="swish", kernel_initializer="he_uniform", name="dense_64_2")(x)
    x = tf.keras.layers.Dense(64, activation="swish", kernel_initializer="he_uniform", name="dense_64_3")(x)
    x = tf.keras.layers.Dense(64, activation="swish", kernel_initializer="he_uniform", name="dense_64_4")(x)
    mass_out = tf.keras.layers.Dense(
        3, activation=lambda x: 2.0 * tf.sigmoid(x), name="masses"
    )(x)
    return tf.keras.Model(inputs=inputs, outputs=mass_out, name="simple_dnn_mass_model")

In [ ]:
# Block 3 — Gauss-Laguerre pressure integral (same as MLQCD.ipynb)
deg = 50
nodes, weights = np.polynomial.laguerre.laggauss(deg)
p_grid = tf.constant(nodes.reshape(1, -1), dtype=tf.float32)
w_grid = tf.constant((weights / np.exp(-nodes)).reshape(1, -1), dtype=tf.float32)

def calculate_pressure(T, MuB, mg, mu, ms):
    T_b = tf.reshape(T, (-1, 1))
    MuB_b = tf.reshape(MuB, (-1, 1))
    mu_q = MuB_b / 3.0

    Eg = tf.sqrt(tf.square(p_grid) + tf.square(mg))
    Eu = tf.sqrt(tf.square(p_grid) + tf.square(mu))
    Es = tf.sqrt(tf.square(p_grid) + tf.square(ms))

    safe_bose = tf.maximum(-tf.math.expm1(-Eg / T_b), 1e-7)
    integrand_g = tf.square(p_grid) * tf.math.log(safe_bose)
    Pg = -(16.0 / (2.0 * np.pi**2)) * T_b * tf.reduce_sum(integrand_g * w_grid, axis=1, keepdims=True)

    log_u_pos = tf.math.softplus(-(Eu - mu_q) / T_b)
    log_u_neg = tf.math.softplus(-(Eu + mu_q) / T_b)
    Pu = (6.0 / (2.0 * np.pi**2)) * T_b * tf.reduce_sum(tf.square(p_grid) * (log_u_pos + log_u_neg) * w_grid, axis=1, keepdims=True)

    log_s_pos = tf.math.softplus(-(Es - mu_q) / T_b)
    log_s_neg = tf.math.softplus(-(Es + mu_q) / T_b)
    Ps = (6.0 / (2.0 * np.pi**2)) * T_b * tf.reduce_sum(tf.square(p_grid) * (log_s_pos + log_s_neg) * w_grid, axis=1, keepdims=True)

    return Pg + 2.0 * Pu + Ps

In [ ]:
# Block 4 — normalization + load YOUR trained weights
current_folder = os.getcwd()
csv_path = os.path.join(current_folder, "Thermo_data_central.csv")
df_train = pd.read_csv(csv_path)
df_train = df_train[df_train["T"] >= (1.1 * Tc)]  # same QGP filter as MLQCD

X_all = df_train[["T", "MuB"]].values
muB_hat = df_train["MuB/T"].values
# train_test_split with 2 arrays → 4 outputs (was wrongly called with 4 arrays → 8 outputs)
X_train, _, _, _ = train_test_split(
    X_all, muB_hat, test_size=0.2, random_state=42, stratify=muB_hat
)

T_min = float(X_train[:, 0].min())
T_max = float(X_train[:, 0].max())
MuB_min = float(X_train[:, 1].min())
MuB_max = float(X_train[:, 1].max())

T_min_tf = tf.constant(T_min, dtype=tf.float32)
T_range_tf = tf.constant(T_max - T_min + 1e-8, dtype=tf.float32)
MuB_min_tf = tf.constant(MuB_min, dtype=tf.float32)
MuB_range_tf = tf.constant(MuB_max - MuB_min + 1e-8, dtype=tf.float32)

def normalize_X(X_np):
    X_norm = X_np.copy().astype(np.float32)
    X_norm[:, 0] = (X_np[:, 0] - T_min) / (T_max - T_min + 1e-8)
    X_norm[:, 1] = (X_np[:, 1] - MuB_min) / (MuB_max - MuB_min + 1e-8)
    return X_norm

# #region agent log — Hypothesis A: confirm fixed train_test_split returns 4 outputs
try:
    _tts_result = train_test_split(
        X_all, muB_hat, test_size=0.2, random_state=42, stratify=muB_hat
    )
    _dblog("train_test_split returned", {"n_outputs": len(_tts_result), "expected": 4}, "A", "Block4-tts")
except Exception as _e:
    _dblog("train_test_split CRASHED", {"error": str(_e)}, "A", "Block4-tts")
# #endregion

model = build_mass_model()
weight_candidates = [
    "mlqcd_model.weights.h5",
    "trained_model.weights.h5",
    os.path.join("checkpoints", "best_weights.weights.h5"),
]
loaded = False
for wpath in weight_candidates:
    if os.path.isfile(wpath):
        model.load_weights(wpath)
        print(f"Loaded weights: {wpath}")
        loaded = True
        break

if not loaded:
    print("No saved weights found — running quick in-notebook training (1500 epochs).")
    print("For full accuracy, train MLQCD.ipynb and run: model.save_weights('mlqcd_model.weights.h5')")

    # ---- compact training (same physics as MLQCD.ipynb) ----
    _Xtrain = tf.convert_to_tensor(
        normalize_X(df_train[["T","MuB"]].values), dtype=tf.float32
    )
    _T_raw   = tf.constant(df_train["T"].values.reshape(-1,1),   dtype=tf.float32)
    _MuB_raw = tf.constant(df_train["MuB"].values.reshape(-1,1), dtype=tf.float32)
    _delta_t = (df_train["TraceA"].values * df_train["T"].values**3).reshape(-1,1)
    _Ytrain  = tf.convert_to_tensor(
        np.concatenate([df_train[["P/T^4","E/T^4","s/T^3","nB/T^3"]].values, _delta_t], axis=1),
        dtype=tf.float32
    )

    def _quick_loss(model, Xb, Tb, MuBb, Yb):
        with tf.GradientTape() as t:
            t.watch(Tb); t.watch(MuBb)
            T_n   = (Tb - T_min_tf) / T_range_tf
            MuB_n = (MuBb - MuB_min_tf) / MuB_range_tf
            masses = model(tf.concat([T_n, MuB_n], axis=1))
            mg_, mu_, ms_ = masses[:,0:1], masses[:,1:2], masses[:,2:3]
            P_ = calculate_pressure(Tb, MuBb, mg_, mu_, ms_)
        s_   = t.gradient(P_, Tb)
        nB_  = t.gradient(P_, MuBb)
        E_   = Tb*s_ - P_ + MuBb*nB_
        Dt_  = (E_ - 3*P_) / Tb
        s_T3 = s_ / Tb**3; nB_T3 = nB_ / Tb**3
        L = (tf.reduce_mean(tf.abs(s_T3  - Yb[:,2:3]))
           + tf.reduce_mean(tf.abs(nB_T3 - Yb[:,3:4]))
           + tf.reduce_mean(tf.abs(Dt_   - Yb[:,4:5])))
        return L

    _opt = tf.keras.optimizers.Adam(1e-3)
    _ds  = tf.data.Dataset.from_tensor_slices((_Xtrain, _T_raw, _MuB_raw, _Ytrain))
    _ds  = _ds.shuffle(512, seed=42).batch(64).prefetch(tf.data.AUTOTUNE)

    for _ep in range(1500):
        for _Xb, _Tb, _MuBb, _Yb in _ds:
            with tf.GradientTape() as _gt:
                _loss = _quick_loss(model, _Xb, _Tb, _MuBb, _Yb)
            _gr = _gt.gradient(_loss, model.trainable_variables)
            _opt.apply_gradients(zip(_gr, model.trainable_variables))
        if (_ep+1) % 300 == 0:
            print(f"  epoch {_ep+1}/1500  loss={_loss.numpy():.5f}")

    model.save_weights("mlqcd_model.weights.h5")
    print("Quick training done. Weights saved to mlqcd_model.weights.h5")
    loaded = True

# #region agent log
_dblog("Block4 done", {"T_min": T_min, "T_max": T_max, "MuB_min": MuB_min,
                       "MuB_max": MuB_max, "weights_loaded": loaded}, "A", "Block4-end")
# #endregion

In [ ]:
# Block 5 — TF emulator (replaces author's PyTorch ResNet emulator)
def emulator(T_values, muB_values):
    """Batch emulator: masses, thermodynamics, dm/dT, dm/dmu, dP/dn (dpn).
    Uses nested GradientTapes so dn_dmu (d^2P/dmuB^2) is computed correctly.
    """
    T   = tf.constant(np.asarray(T_values,   dtype=np.float32).reshape(-1, 1))
    muB = tf.constant(np.asarray(muB_values, dtype=np.float32).reshape(-1, 1))

    # Outer tape tracks muB for the second derivative (dn_dmu = d^2P/dmuB^2).
    # Inner tape is persistent for all first-order gradients.
    with tf.GradientTape() as outer_tape:
        outer_tape.watch(muB)
        with tf.GradientTape(persistent=True) as inner_tape:
            inner_tape.watch(T)
            inner_tape.watch(muB)
            T_norm   = (T   - T_min_tf) / T_range_tf
            MuB_norm = (muB - MuB_min_tf) / MuB_range_tf
            masses = model(tf.concat([T_norm, MuB_norm], axis=1))
            Mg  = masses[:, 0:1]
            Mud = masses[:, 1:2]
            Ms  = masses[:, 2:3]
            P   = calculate_pressure(T, muB, Mg, Mud, Ms)

        s       = inner_tape.gradient(P,   T)
        nB      = inner_tape.gradient(P,   muB)
        dmdT_ud = inner_tape.gradient(Mud, T)
        dmdT_s  = inner_tape.gradient(Ms,  T)
        dmdT_g  = inner_tape.gradient(Mg,  T)
        dmdmu_ud = inner_tape.gradient(Mud, muB)
        dmdmu_s  = inner_tape.gradient(Ms,  muB)
        dmdmu_g  = inner_tape.gradient(Mg,  muB)
        del inner_tape

    # Second derivative: d(nB)/d(muB) = d^2P/d(muB)^2
    dn_dmu = outer_tape.gradient(nB, muB)
    del outer_tape

    # #region agent log — Hypothesis B: verify dn_dmu is NOT None with nested tapes
    _dblog("dn_dmu check", {"is_none": dn_dmu is None, "type": str(type(dn_dmu))}, "B", "Block5-dn_dmu")
    # #endregion

    E     = T * s - P + muB * nB
    trace = E - 3.0 * P

    if dn_dmu is None:
        # Fallback: numerical dP/dn via finite difference on nB
        dPN = tf.zeros_like(nB)
        _dblog("dPN fallback still used (nested tape failed)", {}, "B", "Block5-dpn")
    else:
        dP_dmu = nB                                    # dP/dmuB = nB (by definition)
        dPN    = dP_dmu / tf.maximum(tf.abs(dn_dmu), 1e-20)

    to_np = lambda x: x.numpy().flatten()
    return (
        to_np(Mud), to_np(Ms), to_np(Mg),
        to_np(P), to_np(s), to_np(E), to_np(trace), to_np(nB),
        to_np(T), to_np(muB),
        to_np(dmdT_ud), to_np(dmdT_s), to_np(dmdT_g),
        to_np(dmdmu_ud), to_np(dmdmu_s), to_np(dmdmu_g),
        to_np(dPN),
    )

In [ ]:
# ══════════════════════════════════════════════════════════
# VERIFICATION CELL — proves this is YOUR model, not the author's
# ══════════════════════════════════════════════════════════

print("=" * 60)
print("  MODEL IDENTITY VERIFICATION")
print("=" * 60)

print(f"\n  Framework     : TensorFlow {tf.__version__}  (author uses PyTorch)")
print(f"  Model name    : {model.name}  (author uses Net_Mass)")
print(f"  Total params  : {model.count_params():,}  (author has 3 separate nets ~30k total)")
print(f"  Architecture  : Plain DNN, NO residual blocks")
print(f"  Author uses   : ResNet with residual skip connections")

print("\n  Layer-by-layer (YOUR model):")
for layer in model.layers:
    cfg = layer.get_config()
    act = cfg.get("activation", "—")
    if hasattr(act, "__name__"):
        act = act.__name__
    units = cfg.get("units", "—")
    print(f"    {layer.name:25s}  units={units}  activation={act}")

print("\n  Author's architecture (for comparison):")
print("    input_layer1              units=16  activation=swish(manual)")
print("    hidden_layers0            units=32  activation=swish(manual)")
print("    hidden_layers1..7         ResidualBlock(32) × 7  [SKIP CONNECTIONS]")
print("    hidden_layers8            units=16")
print("    output_layer              units=1  (×3 separate nets)")

print("\n  Weight source used in THIS run:")
for wpath in ["mlqcd_model.weights.h5", "trained_model.weights.h5", "checkpoints/best_weights.weights.h5"]:
    if os.path.isfile(wpath):
        size_kb = os.path.getsize(wpath) / 1024
        print(f"    ✓  {wpath}  ({size_kb:.1f} KB)  — Keras .h5, NOT PyTorch .pt")
        break
else:
    print("    (weights were trained inline in Block 4)")

print("\n  Author's weight files (NOT used here):")
for pt in ["Mud_model.pt", "Ms_model.pt", "Mg_model.pt"]:
    exists = os.path.isfile(pt)
    print(f"    {'✓ EXISTS' if exists else '✗ ABSENT'}  {pt}")

print("=" * 60)
print("  CONCLUSION: This notebook uses YOUR TF plain-DNN,")
print("  NOT the author's PyTorch ResNet.")
print("=" * 60)

In [ ]:
# Block 6 — emulation on full WB grid (same as Bulk_New Block 5)
df = pd.read_csv(csv_path)
Temperature = df["T"].to_numpy().flatten()
Baryon = df["MuB"].to_numpy().flatten()

print(f"Running TF emulator on {len(Temperature)} grid points ...")
(
    mud, ms, mg, pressure, entropy, energy, trace, baryon,
    Temp, Mub, dmdT_ud, dmdT_s, dmdT_g,
    dmdmu_ud, dmdmu_s, dmdmu_g, dpn1,
) = emulator(Temperature, Baryon)
print("Emulation done.")

In [ ]:
# Block 7 — speed of sound c_s^2 (Bulk_New Block 6)
dP_dT = np.gradient(pressure, Temp)
dE_dT = np.gradient(energy, Temp)
Cs2 = dP_dT / np.maximum(np.abs(dE_dT), 1e-30)

plt.figure(figsize=(7, 6))
plt.plot(Temp[46:116] / Tc, Cs2[46:116], marker=',', label=r'$\mu_B/T=0$')
plt.plot(Temp[278:348] / Tc, Cs2[278:348], marker=',', label=r'$\mu_B/T=1$')
plt.plot(Temp[510:580] / Tc, Cs2[510:580], marker=',', label=r'$\mu_B/T=2$')
plt.plot(Temp[742:812] / Tc, Cs2[742:812], marker=',', label=r'$\mu_B/T=3$')
plt.ylabel(r'$C_s^2$', fontsize=14)
plt.xlabel(r'$T/T_c$', fontsize=14)
plt.ylim(0.1, 0.5)
plt.legend(fontsize=14)
plt.title('Variation of Speed of Sound', fontsize=14)
plt.grid(True)
plt.savefig("./plots/cs2_sanchit.jpg")
plt.show()

In [ ]:
# Block 8 — g^2 reference (Plumari WB parametrization, Bulk_New Block 7)
lambda_WB, Ts_Tc_WB = 2.6, 0.57
g2_ref = 48 * np.pi**2 / (((11 * 3) - (2 * 3)) * np.log((lambda_WB * (Temp / Tc - Ts_Tc_WB)) ** 2))

plt.figure(figsize=(7, 6))
plt.plot(Temp[30:116] / Tc, np.sqrt(g2_ref[30:116]), label=r'$g^2_{WB\,(Ref)}$', color='black', linestyle='--')
plt.ylabel(r'$g$', fontsize=14)
plt.xlabel(r'$T/T_c$', fontsize=14)
plt.legend(fontsize=14)
plt.title(r'Variation of $g$ (Comparison)', fontsize=14)
plt.grid(True)
plt.savefig("./plots/g2_sanchit.jpg")
plt.show()

In [ ]:
# Block 9 — relaxation times tau_q, tau_g (Bulk_New Block 8)
nc = 3
kwb = 18.5
# Guard: g2_ref can be negative at low T where log argument is < 1.
# np.log(2*kwb / g2_ref) is undefined for g2_ref <= 0; use nan so those
# rows never contribute to the plotted/integrated QGP region (T >= 1.1 Tc).
_safe_g2  = np.where(g2_ref > 0, g2_ref, np.nan)
_log_arg  = np.where(_safe_g2 > 0, 2 * kwb / _safe_g2, np.nan)
_log_term = np.where(_log_arg > 0, np.log(_log_arg), np.nan)
tauqinv_wb = (nc**2 - 1) / nc * (_safe_g2 * Temp) / (8 * np.pi) * _log_term * GEV_TO_INV_FM
tauginv_wb = 2 * nc * _safe_g2 * Temp / (8 * np.pi) * _log_term * GEV_TO_INV_FM
tauq_wb = 1.0 / np.where(tauqinv_wb > 0, tauqinv_wb, np.nan)
taug_wb = 1.0 / np.where(tauginv_wb > 0, tauginv_wb, np.nan)

# #region agent log — Hypothesis D: NaN/inf in tau arrays from negative g2_ref
_nan_tq = int(np.sum(np.isnan(tauq_wb)))
_nan_tg = int(np.sum(np.isnan(taug_wb)))
_nan_g2 = int(np.sum(np.isnan(g2_ref)))
_neg_g2 = int(np.sum(g2_ref < 0))
_dblog("tau NaN check", {
    "nan_tauq": _nan_tq, "nan_taug": _nan_tg,
    "nan_g2ref": _nan_g2, "neg_g2ref": _neg_g2,
    "g2ref_min": float(np.nanmin(g2_ref)), "g2ref_max": float(np.nanmax(g2_ref)),
    "total_rows": len(Temp)
}, "D", "Block9-tau")
# #endregion

plt.figure(figsize=(7, 6))
plt.plot(Temp[45:116] / Tc, tauq_wb[45:116], label=r'$\tau_{q}$', linestyle='-.', color='red')
plt.plot(Temp[45:116] / Tc, taug_wb[45:116], label=r'$\tau_{g}$', linestyle='--', color='blue')
plt.ylabel(r'$\tau$  [fm/c]', fontsize=18)
plt.xlabel(r'$T/T_c$', fontsize=18)
plt.legend(loc='best', fontsize=14)
plt.tick_params(which='both', direction='in', top=True, right=True)
plt.minorticks_on()
plt.grid(True)
plt.xlim(1.1, 1.55)
plt.savefig("./plots/Tau_sanchit.jpg", dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
# Block 10 — bulk viscosity xi/s (Bulk_New Block 9, scipy quad integrals)
coeff = -(4 * np.pi) / (3 * (2 * np.pi) ** 3)
MUB = Mub / 3.0

def integrand_ud_quark(p, mass_ud, Tmp, tq, Mu, cs2, dmud, dmudmu, dpn):
    En = np.sqrt(p**2 + mass_ud**2)
    fq = 1.0 / (np.exp((En - Mu) / Tmp) + 1.0)
    return (
        p**2 * tq / Tmp * (mass_ud**2 / En) * fq * (1 - fq)
        * (p**2 / (3 * En) - cs2 * (En - (mass_ud / En) * Tmp * dmud - Mu * (mass_ud / En) * dmudmu)
           + dpn * ((mass_ud / En) * dmudmu - 1))
        * 2 * 3 * 2 * GEV_TO_INV_FM
    )

def integrand_ud_anti(p, mass_ud, Tmp, tq, Mu, cs2, dmud, dmudmu, dpn):
    En = np.sqrt(p**2 + mass_ud**2)
    fq = 1.0 / (np.exp((En + Mu) / Tmp) + 1.0)
    return (
        p**2 * tq / Tmp * (mass_ud**2 / En) * fq * (1 - fq)
        * (p**2 / (3 * En) - cs2 * (En - (mass_ud / En) * Tmp * dmud + Mu * (mass_ud / En) * dmudmu)
           + dpn * ((mass_ud / En) * dmudmu - 1))
        * 2 * 3 * 2 * GEV_TO_INV_FM
    )

# Strange-quark integrands: mass_ud added as explicit arg (fixes free-variable NameError/wrong-physics bug)
def integrand_s_quark(p, mass_s, mass_ud_i, Tmp, tq, Mu, cs2, ds, dsmu, dpn):
    En = np.sqrt(p**2 + mass_s**2)
    fq = 1.0 / (np.exp((En - Mu) / Tmp) + 1.0)
    return (
        p**2 * tq / Tmp * (mass_s**2 / En) * fq * (1 - fq)
        * (p**2 / (3 * En) - cs2 * (En - (mass_s / En) * Tmp * ds - Mu * (mass_s / En) * dsmu)
           + dpn * ((mass_ud_i / En) * dsmu - 1))
        * 2 * 3 * GEV_TO_INV_FM
    )

def integrand_s_anti(p, mass_s, mass_ud_i, Tmp, tq, Mu, cs2, ds, dsmu, dpn):
    En = np.sqrt(p**2 + mass_s**2)
    fq = 1.0 / (np.exp((En + Mu) / Tmp) + 1.0)
    return (
        p**2 * tq / Tmp * (mass_s**2 / En) * fq * (1 - fq)
        * (p**2 / (3 * En) - cs2 * (En - (mass_s / En) * Tmp * ds + Mu * (mass_s / En) * dsmu)
           + dpn * ((mass_ud_i / En) * dsmu - 1))
        * 2 * 3 * GEV_TO_INV_FM
    )

def integrand_gluon(p, mass_g, mass_ud_i, Tmp, tg, Mu, cs2, dg, dgmu, dpn):
    En = np.sqrt(p**2 + mass_g**2)
    fg = 1.0 / (np.exp(En / Tmp) - 1.0)
    return (
        p**2 * tg / Tmp * (mass_g**2 / En) * fg * (1 + fg)
        * (p**2 / (3 * En) - cs2 * (En - (mass_g / En) * Tmp * dg) + dpn * ((mass_ud_i / En) * dgmu - 1))
        * 2 * 8 * GEV_TO_INV_FM
    )

print("Computing bulk viscosity integrals (this may take a few minutes) ...")
xi_udi = []
for mass_ud, Tmp, tq, Mu, cs2, dmud, dmudmu, dpn in zip(
    mud, Temp, tauq_wb, MUB, Cs2, dmdT_ud, dmdmu_ud, dpn1
):
    if np.isnan(tq) or np.isnan(cs2):
        xi_udi.append(np.nan); continue
    i1u, _ = quad(integrand_ud_quark, 0, 10, args=(mass_ud, Tmp, tq, Mu, cs2, dmud, dmudmu, dpn))
    i2u, _ = quad(integrand_ud_anti, 0, 10, args=(mass_ud, Tmp, tq, Mu, cs2, dmud, dmudmu, dpn))
    xi_udi.append(i1u + i2u)
xi_ud = np.array(xi_udi)

xi_si = []
for mass_s, mass_ud_i, Tmp, tq, Mu, cs2, ds, dsmu, dpn in zip(
    ms, mud, Temp, tauq_wb, MUB, Cs2, dmdT_s, dmdmu_s, dpn1
):
    if np.isnan(tq) or np.isnan(cs2):
        xi_si.append(np.nan); continue
    i1s, _ = quad(integrand_s_quark, 0, 10, args=(mass_s, mass_ud_i, Tmp, tq, Mu, cs2, ds, dsmu, dpn))
    i2s, _ = quad(integrand_s_anti, 0, 10, args=(mass_s, mass_ud_i, Tmp, tq, Mu, cs2, ds, dsmu, dpn))
    xi_si.append(i1s + i2s)
xi_s = np.array(xi_si)

xi_gi = []
for mass_g, mass_ud_i, Tmp, tg, Mu, cs2, dg, dgmu, dpn in zip(
    mg, mud, Temp, taug_wb, MUB, Cs2, dmdT_g, dmdmu_g, dpn1
):
    if np.isnan(tg) or np.isnan(cs2):
        xi_gi.append(np.nan); continue
    i1g, _ = quad(integrand_gluon, 0, 10, args=(mass_g, mass_ud_i, Tmp, tg, Mu, cs2, dg, dgmu, dpn))
    xi_gi.append(i1g)
xi_g = np.array(xi_gi)

Xi = (xi_ud + xi_s + xi_g) * coeff / np.maximum(np.abs(entropy), 1e-30)

# #region agent log — Hypothesis C/D/E: Xi array stats
_dblog("Xi computed", {
    "len_Xi": len(Xi), "len_Temp": len(Temp),
    "nan_Xi": int(np.sum(np.isnan(Xi))),
    "inf_Xi": int(np.sum(np.isinf(Xi))),
    "Xi_45_116_min": float(np.nanmin(Xi[45:116])),
    "Xi_45_116_max": float(np.nanmax(Xi[45:116])),
    "index_742_in_bounds": len(Xi) > 812,
}, "E", "Block10-Xi")
# #endregion

print("Bulk viscosity computed.")

In [ ]:
# Block 11 — plot xi/s and save CSV (same slices as Bulk_New)
plt.figure(figsize=(10, 6))
plt.plot(Temp[45:116] / Tc, Xi[45:116], label=r'$DNN$ $\hat{\mu} = 0$', marker='.', color='red')
plt.plot(Temp[277:348] / Tc, Xi[277:348], label=r'$DNN$ $\hat{\mu} = 1$', marker='.', color='b')
plt.plot(Temp[509:580] / Tc, Xi[509:580], label=r'$DNN$ $\hat{\mu} = 2$', marker='.', color='g')
plt.plot(Temp[741:812] / Tc, Xi[741:812], label=r'$DNN$ $\hat{\mu} = 3$', marker='.', color='y')
plt.ylabel(r'$\xi/s$', fontsize=20)
plt.xlabel(r'$T/T_c$', fontsize=20)
plt.legend(loc='best')
plt.grid(True)
plt.title(r'Variation of $\xi/s$ with $k=18$ (Sanchit TF DNN)')
plt.savefig("./plots/Xi_s_all_sanchit.jpg")
plt.show()

pd.DataFrame({"T": Temp, "Xi_mid": Xi}).to_csv("bulk_mid_sanchit.csv", index=False)
print("Saved: plots/Xi_s_all_sanchit.jpg, bulk_mid_sanchit.csv")